In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

d:\Git_Repositories\Attention\SelfAttention\.venv\lib\site-packages\torch\_subclasses\functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
class SlidingWindowAttention(nn.Module):
    def __init__(self, k, heads=8, window_size=3):
        super().__init__()
        
        assert k % heads == 0, "Embedding dim (k) must be divisible by heads."
        assert window_size % 2 == 1, "Window size must be an odd number."
        

        self.k, self.heads, self.window_size = k, heads, window_size
        self.s = k // heads  
        

        self.to_queries = nn.Linear(k, k, bias=False)
        self.to_keys = nn.Linear(k, k, bias=False)
        self.to_values = nn.Linear(k, k, bias=False)
        self.unifyheads = nn.Linear(k, k)

    def forward(self, x):
        b, t, k = x.size()
        h = self.heads
        s = self.s 


        queries = self.to_queries(x).view(b, t, h, s)
        keys = self.to_keys(x).view(b, t, h, s)
        values = self.to_values(x).view(b, t, h, s)


        queries = queries.transpose(1, 2).contiguous().view(b * h, t, s)
        keys = keys.transpose(1, 2).contiguous().view(b * h, t, s)
        values = values.transpose(1, 2).contiguous().view(b * h, t, s)

        dot = torch.bmm(queries, keys.transpose(1, 2)) # Shape: (b*h, t, t)

        #key difference from the standard attention
        # indices for the sequence length dimension
        row_indices = torch.arange(t, device=x.device).view(1, -1)
        col_indices = torch.arange(t, device=x.device).view(-1, 1)


        # Calculate distance and create a boolean mask
        # The mask is True for positions within the window
        distance = torch.abs(row_indices - col_indices)
        window_mask = distance <= self.window_size // 2 # Shape: (t, t)


        # Apply the mask. We add -infinity to positions outside the window.
        # This makes their softmax score effectively zero.
        dot.masked_fill_(~window_mask, float('-inf'))

        
        dot = dot / (s ** 0.5) # Scale
        attn_weights = F.softmax(dot, dim=2) 

        output = torch.bmm(attn_weights, values).view(b, h, t, s)
        output = output.transpose(1, 2).contiguous().view(b, t, k)

        return self.unifyheads(output)

In [ ]:
batch_size = 4
seq_length = 1024 # Can be much longer now
embedding_dim = 256
num_heads = 8
window = 31 # Must be odd


x = torch.randn(batch_size, seq_length, embedding_dim)


sliding_window_attn = SlidingWindowAttention(
    k=embedding_dim,
    heads=num_heads,
    window_size=window
)
output = sliding_window_attn(x)

print(f"Input shape: {x.shape}")
print(f"Sliding window attention output shape: {output.shape}")

assert output.shape == x.shape

Input shape: torch.Size([4, 1024, 256])
Sliding window attention output shape: torch.Size([4, 1024, 256])
